# L1001 · Browser translation
Run through l1001.html. The bridge supplies inputs in memory and executes this published notebook in a temporary Pyodide session. Never paste an API key into a notebook cell.


In [ ]:
import importlib, json, sys
from pathlib import Path
from pyodide.http import pyfetch
from IPython.display import display
import micropip

print("Loading L1001 browser dependencies…", flush=True)
await micropip.install("striprtf==0.0.29")
# The bridge fixes this URL to this deployment's published files.
runtime_dir = Path("/tmp/l1001-runtime")
runtime_dir.mkdir(exist_ok=True)
sys.path.insert(0, str(runtime_dir))
for name in ("l1001_core.py", "l1001_browser.py"):
    response = await pyfetch(L1001_ASSET_BASE + name, cache="no-store")
    if not response.ok:
        raise RuntimeError("Cannot load the published L1001 Python module: " + name)
    (runtime_dir / name).write_text(await response.string(), encoding="utf-8")
importlib.invalidate_caches()
from l1001_browser import run

def emit(payload):
    display({"application/vnd.l1001+json": json.loads(json.dumps(payload))}, raw=True)

try:
    outputs = await run(inputs, emit)
finally:
    inputs.clear()
